In [2]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [3]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/Carenza2023_Part2.h5ad")

In [4]:
adata = adata[adata.obs['donor_id'].isin(["GLIO1_T", "GLIO2_T", "GLIO3_T", "GLIO4_T", "GLIO5_T", "GLIO6_T", "GLIO7_T"])]

In [5]:
df_obs = pd.DataFrame(adata.obs)

In [6]:
del adata.obs

In [7]:
adata = adata.raw.to_adata()

In [8]:
adata

AnnData object with n_obs × n_vars = 23430 × 17679
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_features'
    uns: 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [9]:
#adata = adata.raw.to_adata()

In [10]:
X_counts_recovered, size_factors_sub=ov.pp.recover_counts(adata.X, 50*1e4, 50*1e5, log_base=None, 
                                                          chunk_size=10000)


100%|██████████| 3430/3430 [00:04<00:00, 787.32it/s]


In [11]:
adata.X = X_counts_recovered

In [12]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [13]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [14]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [15]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [16]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
AL627309.1,False,19,0.081093,True,0.000811,0.000810,0.980248
AL669831.5,False,379,1.617584,True,0.016389,0.016548,1.146501
FAM87B,False,102,0.435339,True,0.004396,0.004462,1.212876
LINC00115,False,556,2.373026,True,0.024541,0.025647,1.250176
FAM41C,False,500,2.134016,True,0.021810,0.022274,1.084273
...,...,...,...,...,...,...,...
AC011043.1,False,21,0.089629,True,0.000939,0.001023,0.731907
AL592183.1,False,361,1.540760,True,0.015706,0.016058,0.990910
AC007325.4,False,162,0.691421,True,0.007256,0.007972,0.812248
AL354822.1,False,146,0.623133,True,0.006359,0.006575,1.236781


In [17]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [24]:
df_tmp = df_tmp.reset_index().drop_duplicates(['index']).set_index(['index'])

In [25]:
adata.var = df_tmp

In [26]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [27]:
adata

View of AnnData object with n_obs × n_vars = 23430 × 14162
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'ensembl_gene_id', 'start_position', 'end_position', 'chromosome_name'
    uns: 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [28]:
adata.obs['donor_id'] = df_obs['donor_id']

In [29]:
metadata_data = {
    'Author': ['Carenza2023'] * 7,
    'donor_id': ["GLIO1_T", "GLIO2_T", "GLIO3_T", "GLIO4_T", "GLIO5_T", "GLIO6_T", "GLIO7_T"],
    'stage': ['Primary'] * 7,
    'assay': ['10x 3\' v2'] * 7,
    'tissue': ['brain'] * 7,
    'Cells': ['CD45'] * 7,
    'Method': ['cell'] * 7
}

metadata_df = pd.DataFrame(metadata_data)

# Display the metadata DataFrame
print(metadata_df)

        Author donor_id    stage      assay tissue Cells Method
0  Carenza2023  GLIO1_T  Primary  10x 3' v2  brain  CD45   cell
1  Carenza2023  GLIO2_T  Primary  10x 3' v2  brain  CD45   cell
2  Carenza2023  GLIO3_T  Primary  10x 3' v2  brain  CD45   cell
3  Carenza2023  GLIO4_T  Primary  10x 3' v2  brain  CD45   cell
4  Carenza2023  GLIO5_T  Primary  10x 3' v2  brain  CD45   cell
5  Carenza2023  GLIO6_T  Primary  10x 3' v2  brain  CD45   cell
6  Carenza2023  GLIO7_T  Primary  10x 3' v2  brain  CD45   cell


In [30]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

      donor_id       Author    stage      assay tissue Cells Method
0      GLIO1_T  Carenza2023  Primary  10x 3' v2  brain  CD45   cell
1      GLIO1_T  Carenza2023  Primary  10x 3' v2  brain  CD45   cell
2      GLIO1_T  Carenza2023  Primary  10x 3' v2  brain  CD45   cell
3      GLIO1_T  Carenza2023  Primary  10x 3' v2  brain  CD45   cell
4      GLIO1_T  Carenza2023  Primary  10x 3' v2  brain  CD45   cell
...        ...          ...      ...        ...    ...   ...    ...
23425  GLIO7_T  Carenza2023  Primary  10x 3' v2  brain  CD45   cell
23426  GLIO7_T  Carenza2023  Primary  10x 3' v2  brain  CD45   cell
23427  GLIO7_T  Carenza2023  Primary  10x 3' v2  brain  CD45   cell
23428  GLIO7_T  Carenza2023  Primary  10x 3' v2  brain  CD45   cell
23429  GLIO7_T  Carenza2023  Primary  10x 3' v2  brain  CD45   cell

[23430 rows x 7 columns]


In [31]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [33]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
GLIO1_T-0,GLIO1_T,558,1160.0,Non-neoplastic,,,Microglial cell,Langerhans Cells,unknown
GLIO1_T-1,GLIO1_T,952,2443.0,Non-neoplastic,,,Microglial cell,NK Cells,unknown
GLIO1_T-2,GLIO1_T,911,2287.0,Non-neoplastic,,,Microglial cell,Macrophages,unknown
GLIO1_T-3,GLIO1_T,564,1089.0,Non-neoplastic,,,Microglial cell,Langerhans Cells,unknown
GLIO1_T-4,GLIO1_T,902,1566.0,Non-neoplastic,,,Microglial cell,Microglia,unknown
...,...,...,...,...,...,...,...,...,...
GLIO7_T-4518,GLIO7_T,549,1387.0,Non-neoplastic,,,T cell,T Cells,unknown
GLIO7_T-4519,GLIO7_T,758,2135.0,Non-neoplastic,,,T cell,T Cells,unknown
GLIO7_T-4524,GLIO7_T,1085,3252.0,Non-neoplastic,,,T cell,T Cells,unknown
GLIO7_T-4527,GLIO7_T,1064,2221.0,Non-neoplastic,,,Pericyte,Pericytes,unknown


In [34]:
merged_obs_df.index= df_obs.index

In [35]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [36]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [37]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [38]:
del merged_obs_df['donor_id_y']

In [39]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [40]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [41]:
adata.obs = merged_obs_df

In [42]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    686 total control genes are used. (0:00:00)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    944 total control genes are used. (0:00:00)
-->     'phase', cell cycle phase (adata.obs)


In [43]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/Carenza2023_Part3.h5ad")